# Note that this project takes place in R

<center><img src="bicycles.jpg"></center><br>


Toronto, Ontario 🌆. The Queen City. The 6ix.

Known for its vibrant arts scene, diverse culture, stunning skyline, and bustling neighborhoods, Toronto is a city that never sleeps. However, as with any major urban center, it faces its share of challenges. One growing concern for Torontonians is the rising number of bicycle thefts.

You have been invited to assist the Toronto Police Service by analyzing and visualizing data to uncover patterns in theft activity. Your findings and visual insights will provide crucial information that can help allocate resources more effectively and develop strategies to combat bike thefts, ensuring a safer city for all cyclists.

## The Data

The dataset used for analyzing bike thefts is titled `Cleaned_Bicycle_Thefts_Open_Data.csv` in the `data` folder. This dataset contains essential information regarding bicycle theft incidents in a given city. Below are the details of each column in the dataset:

| Column     | Description              |
|------------|--------------------------|
| `date` | The date when the bike theft occurred, formatted as YYYY/MM/DD. | 
| `quarter` | A quarter represents one-fourth (1/4) of a year, equating to three months. |
| `day_of_week` | The day of the week when the theft took place (e.g., Monday, Tuesday). |
| `neighborhood` | The neighborhood where the theft occurred, based on the city's 140 social planning neighborhoods. |
| `bike_cost` | The reported cost of the stolen bike, specified in the local currency. |
| `location` | The specific location type of the theft, such as Residential Structures, Commercial Areas, Public Spaces, etc. |
| `long` | The longitude of the center of the neighborhood. |
| `lat` | The latitude of the center of the neighborhood. |

This dataset provides a comprehensive view of bike thefts, including when and where they occur, the financial impact, and other spatial and temporal factors. By analyzing this data, you can gain valuable insights into patterns and trends, which can inform strategies to mitigate bike thefts and enhance urban planning.


In [88]:
## Load tidyverse package
library(tidyverse)
library(stringr)

In [89]:
## Read `bike_data`
bike_data <- read_csv("data/Cleaned_Bicycle_Thefts_Open_Data.csv")
## Take a glance of the `bike_data`
head(bike_data)

Rows: 31833 Columns: 8
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (3): day_of_week, neighborhood, location
dbl  (3): bike_cost, long, lat
date (2): date, quarter

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


date,quarter,day_of_week,neighborhood,bike_cost,location,long,lat
<date>,<date>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>
2014-01-01,2014-01-01,Wednesday,Waterfront Communities-The Island (77),1019,Residential Structures,-79.37720,43.63388
2014-01-03,2014-01-01,Friday,Humewood-Cedarvale (106),560,Residential Structures,-79.42768,43.69137
2014-01-04,2014-01-01,Saturday,Church-Yonge Corridor (75),200,Residential Structures,-79.37902,43.65965
2014-01-04,2014-01-01,Saturday,Leaside-Bennington (56),900,Residential Structures,-79.36607,43.70380
2014-01-07,2014-01-01,Tuesday,South Riverdale (70),150,Residential Structures,-79.33565,43.64929
2014-01-08,2014-01-01,Wednesday,Moss Park (73),479,Residential Structures,-79.36730,43.65652


# Q1
Which quarter, i.e., `"Q1"`, `"Q2"`, `"Q3"`, and `"Q4"`, has the highest and lowest number of stolen bikes? Store your findings as string variables `high` and `low`.

In [91]:
#Q1
    #note that the data ranges from 2014-01-01 - 2023-12-31
    #first need to identify what quarter each date is in, because the original "quarter" column does this for each year; 
        #want to determine quarter separately from year (Q1/Q2/Q3/Q4)

q1_data <- bike_data %>%
    #create a "new_quarter" column to determine quarter regardless of year
    mutate(new_quarter = case_when(str_detect(quarter, "01-01") ~ "Q1",
                                   str_detect(quarter, "04-01") ~ "Q2",
                                   str_detect(quarter, "07-01") ~ "Q3",
                                   str_detect(quarter, "10-01") ~ "Q4")) %>%
    group_by(new_quarter) %>%
    summarize(num_stolen = n()) %>%
    arrange(num_stolen)
print(q1_data)

#assign quarters with most/least bike thefts
high <- q1_data$new_quarter[4]
low <- q1_data$new_quarter[1]
print(high)
print(low)

# A tibble: 4 × 2
  new_quarter num_stolen
  <chr>            <int>
1 Q1                2890
2 Q2                5003
3 Q4               10614
4 Q3               13326
[1] "Q3"
[1] "Q1"


In [ ]:
#make a plot
#make a plot
q1_plot_I <- ggplot(data=q1_data, aes(x=new_quarter, y=num_stolen)) + geom_col() +
  labs(title="Number of Stolen Bikes in Toronto per Quarter (2014-2023)", x="Quarter", 
       y="# Bikes Stolen") + scale_y_continuous(breaks=seq(2500,15000,by=2500)) +
  theme(plot.title=element_text(size=11), axis.title.x=element_text(size=10),
        axis.title.y=element_text(size=10))
q1_plot_I

# Q2
What are the most frequent locations (e.g., Residential, Commercial Areas) for bike thefts in Toronto? And what proportion it is (round to one decimal place)? Store your findings as a string variable, `location` and a numerical variable, `percentage`.

In [92]:
#Q2
    #first, find # total observations in data -- 31,833
q2_data <- bike_data %>%
    group_by(location) %>%
    #calculate # observations per "location" & their proportion of the total # observations (31,833)
        #round these proportions to 1 decimal
    summarize(num = n(), perc = round((num/31833), 1)) %>%
    arrange(desc(num))
print(q2_data)

#assign location with most bike thefts & its proportion of the total # observations
location <- q2_data$location[1]
percentage <- q2_data$perc[1]
print(location)
print(percentage)

# A tibble: 7 × 3
  location                 num  perc
  <chr>                  <int> <dbl>
1 Residential Structures 15120   0.5
2 Commercial Areas        8246   0.3
3 Open/Public Spaces      6424   0.2
4 Transportation           714   0  
5 Public Institutions      668   0  
6 Others                   538   0  
7 Group/Shared Home        123   0  
[1] "Residential Structures"
[1] 0.5


In [ ]:
#make a plot
q2_plot_I <- ggplot(data=q2_data, aes(x=location, y=num)) + geom_col() +
  labs(title="Number of Stolen Bikes in Toronto per Location (2014-2023)", x="Location",
       y="# Bikes Stolen") + scale_y_continuous(breaks=seq(2500,15000,2500)) +
  theme(plot.title=element_text(size=11), axis.title.x=element_text(size=9),
        axis.title.y=element_text(size=9), axis.text.x=element_text(size=7,angle=45, hjust=1))
q2_plot_I

# Q3
In which region of Toronto is the _median_ value of stolen bikes the highest? Store your findings as a string variable, `region` (`region` can be a real region name or a region code, i.e., '1', '2', '3', ...).

In [93]:
#Q3
    #it is unclear, but I assume that region is equivalent to the "neighorhood" column which includes the name of the 
        #neighborhood & a numerical identifier in parentheses; should separate these two identifiers
q3_data <- bike_data %>%
    group_by(neighborhood) %>%
    #get total # thefts & median of "bike_cost"
    summarize(num_thefts = n(), med_bike_cost = median(bike_cost)) %>%
    #sort by median bike cost in descending order
    arrange(desc(med_bike_cost)) %>%
    #split the original "neighborhood" column into 2 to separate the neighborhood id from neighborhood
    mutate(neighborhood_id = str_extract(neighborhood, "([0-9]+)"),
          neighborhood = str_replace(neighborhood, "\\([0-9]+\\)", "")) %>%
    #alter the order of the columns
    select(neighborhood, neighborhood_id, num_thefts, med_bike_cost)
print(q3_data)

#assign the neighborhood/region with the highest median value of "bike_cost"
region <- q3_data$neighborhood[1]
print(region)

# A tibble: 140 × 4
   neighborhood                         neighborhood_id num_thefts med_bike_cost
   <chr>                                <chr>                <int>         <dbl>
 1 "Bridle Path-Sunnybrook-York Mills " 41                      47         1186 
 2 "Bedford Park-Nortown "              39                     106          990.
 3 "Leaside-Bennington "                56                     234          870.
 4 "Edenbridge-Humber Valley "          9                       40          837 
 5 "Alderwood "                         20                      47          800 
 6 "Danforth East York "                59                     106          800 
 7 "Lansing-Westgate "                  38                      68          800 
 8 "Mimico (includes Humber Bay Shores… 17                     474          800 
 9 "Mount Pleasant West "               104                    503          800 
10 "Rosedale-Moore Park "               98                     579          800 
# ℹ 130 

In [ ]:
#Make a plot with the top-5 neighborhoods
#find these neighborhoods
q3_data_sub <- q3_data %>%
    slice_max(order_by=med_bike_cost, n=5) %>%
  #some locations have same 'med_bike_cost'
  head(5)
q3_data_sub

#plot
q3_plot_I <- ggplot(data=q3_data_sub, aes(x=neighborhood, y=med_bike_cost)) + geom_col() +
  geom_text(aes(label=num_thefts), vjust=-0.3, size=3.5) +
  labs(title="Costs of Stolen Bikes in Toronto Neighborhoods (2014-2023)", x="Neighborhood",
       y="Median Bike Cost ($)") +
  scale_y_continuous(breaks=seq(200,1200,200)) +
  theme(plot.title=element_text(size=11), axis.title.x=element_text(size=10),
        axis.title.y=element_text(size=10), 
        axis.text.x=element_text(size=8, angle=25, hjust=0.8, vjust=0.9))
q3_plot_I

# Q4
What course of action would you recommend to the police station based on your findings? Store your recommendation as a string variable, `action`.

**In regards to Q1:**
- Bikes are stolen in quarter 3 significantly more than during any other quarter. Police should be the most wary of bike theft occurring between July 1 through October 1 than any other quarter. They should be least wary between January 1 through April 1. This is not to say they should stop doing their jobs, but police should be more wary in quarters 3 & 4 of bike theft than in quarter 1.

**In regards to Q2:**
- There are 3 locations in which bike thefts mainly occur. In fact, these locations--Residential structures, Commercial areas, Open/Public spaces--accounted for about 93% of all bike thefts in this dataset. Furthermore, nearly half of all bike thefts in Toronto occurred in residential-structure locations. As such, police should implement more methods to protect against bike theft in these 3 locations, particularly residential structures, to better ward against bike thieves.

**In regards to Q3:**
- In the Bridle Path-Sunnybrook-York Mills neighborhood, the median cost of bikes that are stolen is about 200 currency units--presumably Canadian dollars--greater than any other neighborhood. More specifically, of 47 bike thefts that occurred in this neighborhood, more than half were worth at least 1,000 units. This does not indicate that more bike thefts occurred in this neighborhood; however, this could indicate that some people in this neighborhood have very expensive bikes relatively speaking. As such, bike theft in this neighborhood is more costly than in any other, & so police should have a considerable presence in this area. This does not mean that other neighborhoods with a less median bike cost should be disregarded, but this data simply emphasizes the cost of bike theft in this neighborhood.

`action` just superposes these 3 paragraphs into a single string.